# 01 — Data Quality

## Restaurant Customer Intelligence

This notebook audits and prepares the customer dataset used in the project.

### Objectives

- Assess the structure and overall quality of the raw dataset.
- Identify missing, duplicated, inconsistent, and invalid values.
- Investigate relationships between potentially inconsistent variables before applying corrections.
- Define documented data-quality rules.
- Preserve the raw dataset without modification.
- Generate a clean dataset for subsequent exploratory analysis and modeling.

> Cleaning decisions are based on observed data patterns and business logic rather than automatic deletion or imputation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
RAW_PATH = Path("../data/raw/base_datos_restaurantes_USA_v2.csv")

df_raw = pd.read_csv(RAW_PATH)

# Working copy: raw data remains unchanged
df = df_raw.copy()

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 30,000
Columns: 17


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id_persona                 30000 non-null  int64  
 1   nombre                     30000 non-null  str    
 2   apellido                   30000 non-null  str    
 3   edad                       29899 non-null  float64
 4   genero                     30000 non-null  str    
 5   ciudad_residencia          30000 non-null  str    
 6   estrato_socioeconomico     30000 non-null  str    
 7   frecuencia_visita          30000 non-null  int64  
 8   promedio_gasto_comida      29855 non-null  float64
 9   ocio                       30000 non-null  str    
 10  consume_licor              30000 non-null  str    
 11  preferencias_alimenticias  28597 non-null  str    
 12  membresia_premium          30000 non-null  str    
 13  telefono_contacto          14834 non-null  str    
 14  c

In [4]:
df.head()

,id_persona,nombre,apellido,edad,genero,ciudad_residencia,estrato_socioeconomico,frecuencia_visita,promedio_gasto_comida,ocio,consume_licor,preferencias_alimenticias,membresia_premium,telefono_contacto,correo_electronico,tipo_de_pago_mas_usado,ingresos_mensuales
0,2550327378,Jackson,Gomez,31.00,Masculino,Miami,Alto,6,67.51,Sí,No,Vegetariano,Sí,(830)220-1926,NaN,Efectivo,6425
1,9446112038,Samantha,Soto,40.00,Femenino,Denver,Medio,2,44.92,Sí,Sí,Mariscos,No,881-476-1426,NaN,Efectivo,2374
2,3098363243,Terry,Adams,62.00,Femenino,Denver,Bajo,2,9.24,Sí,Sí,Vegetariano,No,NaN,diana74@example.net,Efectivo,1110
3,4013002847,James,Shannon,41.00,Masculino,Boston,Alto,5,30.74,Sí,Sí,Carnes,Sí,NaN,scottfrey@example.com,Tarjeta,6931
4,7372911048,Susan,Jones,49.00,Femenino,San Diego,Bajo,0,0.00,No,No,Carnes,No,243.248.8919,glassgary@example.org,Tarjeta,1350


In [5]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id_persona,"30,000.00",NaN,NaN,NaN,"5,504,764,563.82","2,602,799,263.54","1,000,153,386.00","3,243,617,026.00","5,515,865,060.00","7,754,425,564.00","9,999,627,377.00"
nombre,30000,690,Michael,678,NaN,NaN,NaN,NaN,NaN,NaN,NaN
apellido,30000,1000,Smith,636,NaN,NaN,NaN,NaN,NaN,NaN,NaN
edad,"29,899.00",NaN,NaN,NaN,49.67,23.84,-5.00,33.00,49.00,65.00,300.00
genero,30000,2,Femenino,15044,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ciudad_residencia,30000,10,Chicago,5384,NaN,NaN,NaN,NaN,NaN,NaN,NaN
estrato_socioeconomico,30000,4,Medio,9325,NaN,NaN,NaN,NaN,NaN,NaN,NaN
frecuencia_visita,"30,000.00",NaN,NaN,NaN,3.90,2.74,-3.00,2.00,4.00,5.00,10.00
promedio_gasto_comida,"29,855.00",NaN,NaN,NaN,32.60,26.40,0.00,13.29,25.51,44.40,149.97
ocio,30000,2,No,15094,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Visit frequency and food spending consistency

Before applying any cleaning rule, we investigate the relationship between
`frecuencia_visita` and `promedio_gasto_comida`.

Special attention is given to:
- Zero visit frequency with recorded spending.
- Negative visit frequency.
- Zero food spending.
- Negative food spending.

The objective is to distinguish plausible zero values from data-entry errors.

In [6]:
# Frequency distribution, including zero and negative values
df["frecuencia_visita"].value_counts(dropna=False).sort_index()

frecuencia_visita
-3     1547
 0     1463
 1     1459
 2     3659
 3     3697
 4     5042
 5     5932
 6     3746
 7      878
 8      862
 9      839
 10     876
Name: count, dtype: int64

In [7]:
# Spending summary for customers with zero visit frequency
df.loc[
    df["frecuencia_visita"] == 0,
    "promedio_gasto_comida"
].describe()

count   1,463.00
mean        0.00
std         0.00
min         0.00
25%         0.00
50%         0.00
75%         0.00
max         0.00
Name: promedio_gasto_comida, dtype: float64

In [8]:
freq_zero = df[df["frecuencia_visita"] == 0]

pd.Series({
    "frequency_zero_total": len(freq_zero),
    "spending_positive": (freq_zero["promedio_gasto_comida"] > 0).sum(),
    "spending_zero": (freq_zero["promedio_gasto_comida"] == 0).sum(),
    "spending_negative": (freq_zero["promedio_gasto_comida"] < 0).sum(),
    "spending_missing": freq_zero["promedio_gasto_comida"].isna().sum()
})

frequency_zero_total    1463
spending_positive          0
spending_zero           1463
spending_negative          0
spending_missing           0
dtype: int64

### Finding: zero visit frequency

There are 1,463 customers with `frecuencia_visita = 0`.

All 1,463 records also have `promedio_gasto_comida = 0`, with no positive,
negative, or missing spending values.

This consistent relationship suggests that zero visit frequency represents
customers with no recorded restaurant activity rather than a data-quality error.

**Decision:** retain zero values in both variables. These customers may represent
an inactive customer segment and could be relevant for subsequent segmentation
and reactivation strategies.

In [9]:
pd.Series({
    "negative_frequency": (df["frecuencia_visita"] < 0).sum(),
    "negative_spending": (df["promedio_gasto_comida"] < 0).sum(),
    "zero_frequency": (df["frecuencia_visita"] == 0).sum(),
    "zero_spending": (df["promedio_gasto_comida"] == 0).sum()
})

negative_frequency    1547
negative_spending        0
zero_frequency        1463
zero_spending         1463
dtype: int64

In [10]:
df.loc[
    df["frecuencia_visita"] < 0,
    ["frecuencia_visita", "promedio_gasto_comida"]
].describe()

,frecuencia_visita,promedio_gasto_comida
count,"1,547.00","1,537.00"
mean,-3.00,33.59
std,0.00,26.50
min,-3.00,5.01
25%,-3.00,13.90
50%,-3.00,26.13
75%,-3.00,44.24
max,-3.00,145.95


In [11]:
df.loc[
    df["frecuencia_visita"] < 0,
    ["frecuencia_visita", "promedio_gasto_comida"]
].describe()

,frecuencia_visita,promedio_gasto_comida
count,"1,547.00","1,537.00"
mean,-3.00,33.59
std,0.00,26.50
min,-3.00,5.01
25%,-3.00,13.90
50%,-3.00,26.13
75%,-3.00,44.24
max,-3.00,145.95


In [12]:
df.loc[
    df["frecuencia_visita"] < 0,
    "frecuencia_visita"
].value_counts().sort_index()

frecuencia_visita
-3    1547
Name: count, dtype: int64

In [13]:
df.loc[
    df["frecuencia_visita"] == -3,
    "promedio_gasto_comida"
].describe()

count   1,537.00
mean       33.59
std        26.50
min         5.01
25%        13.90
50%        26.13
75%        44.24
max       145.95
Name: promedio_gasto_comida, dtype: float64

In [14]:
df.groupby("frecuencia_visita")["promedio_gasto_comida"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

,count,mean,median,min,max
frecuencia_visita,,,,,
-3,1537,33.59,26.13,5.01,145.95
0,1463,0.00,0.00,0.00,0.00
1,1453,11.70,11.23,5.01,24.46
2,3640,19.23,15.86,5.02,49.92
3,3678,18.99,15.59,5.02,49.53
4,5020,31.94,28.95,5.04,79.99
5,5895,37.22,32.22,5.00,149.73
6,3723,43.71,40.48,5.06,147.86
7,876,63.77,62.48,5.13,148.43


In [15]:
neg_freq = df[df["frecuencia_visita"] == -3]

pd.DataFrame({
    "negative_freq": neg_freq["ciudad_residencia"].value_counts(),
    "total_city": df["ciudad_residencia"].value_counts()
}).assign(
    percentage=lambda x: 
        (x["negative_freq"] / x["total_city"] * 100).round(2)
).sort_values("percentage", ascending=False)

,negative_freq,total_city,percentage
ciudad_residencia,,,
NYC,264,4769,5.54
Miami,173,3186,5.43
San Diego,164,3075,5.33
Chicago,286,5384,5.31
Seattle,114,2191,5.20
Houston,110,2212,4.97
Dallas,128,2602,4.92
Boston,123,2547,4.83
Phoenix,70,1511,4.63


In [16]:
df.loc[
    df["frecuencia_visita"] == -3,
    [
        "ciudad_residencia",
        "estrato_socioeconomico",
        "frecuencia_visita",
        "promedio_gasto_comida",
        "membresia_premium",
        "ingresos_mensuales"
    ]
].sample(15, random_state=42)

,ciudad_residencia,estrato_socioeconomico,frecuencia_visita,promedio_gasto_comida,membresia_premium,ingresos_mensuales
821,Houston,Medio,-3,29.23,No,2370
15467,Phoenix,Alto,-3,64.81,No,5953
20347,Phoenix,Medio,-3,13.03,No,2411
11219,Boston,Muy Alto,-3,54.37,Sí,16332
10302,Houston,Bajo,-3,12.21,No,1223
1693,Phoenix,Alto,-3,53.85,No,5222
7267,NYC,Muy Alto,-3,93.54,No,13945
28263,Houston,Medio,-3,42.57,No,2325
15547,Miami,Medio,-3,36.19,No,3394
20563,NYC,Alto,-3,74.12,Sí,5755


In [17]:
df.groupby("frecuencia_visita").agg(
    gasto_promedio=("promedio_gasto_comida", "mean"),
    ingreso_promedio=("ingresos_mensuales", "mean"),
    premium_pct=("membresia_premium", lambda x: (x == "Sí").mean() * 100)
).round(2)

,gasto_promedio,ingreso_promedio,premium_pct
frecuencia_visita,,,
-3,33.59,"5,335.80",41.69
0,0.00,"1,201.33",0.34
1,11.70,"1,196.90",0.55
2,19.23,"2,012.22",8.69
3,18.99,"2,006.60",8.76
4,31.94,"4,629.48",44.82
5,37.22,"5,990.64",53.10
6,43.71,"7,857.78",76.08
7,63.77,"13,600.35",94.99


### Finding: negative visit frequency

A total of 1,547 records contain `frecuencia_visita = -3`.

The anomaly is distributed consistently across all ten cities, affecting approximately
4.6%–5.5% of customers in each market. This suggests a systematic data-generation
or data-entry issue rather than a city-specific pattern.

The negative sign was initially considered a possible sign-entry error. However,
customers with `frecuencia_visita = -3` differ substantially from customers with
`frecuencia_visita = 3`:

- Mean spending: 33.59 vs. 18.99
- Mean income: 5,335.80 vs. 2,006.60
- Premium membership: 41.69% vs. 8.76%

Their profile is closer to customers with visit frequencies between 4 and 5.

Therefore, converting `-3` to `3` using an absolute-value transformation is not
supported by the observed data.

**Decision:** treat `-3` as an invalid frequency with an unknown true value and
replace it with `NaN`. Customer records will be retained.

In [18]:
# Replace invalid negative frequency with missing value
df.loc[df["frecuencia_visita"] < 0, "frecuencia_visita"] = np.nan

In [19]:
pd.Series({
    "negative_frequency_after_cleaning": (df["frecuencia_visita"] < 0).sum(),
    "missing_frequency_after_cleaning": df["frecuencia_visita"].isna().sum(),
    "zero_frequency_retained": (df["frecuencia_visita"] == 0).sum()
})

negative_frequency_after_cleaning       0
missing_frequency_after_cleaning     1547
zero_frequency_retained              1463
dtype: int64

In [20]:
pd.Series({
    "missing_spending": df["promedio_gasto_comida"].isna().sum(),
    "negative_spending": (df["promedio_gasto_comida"] < 0).sum(),
    "zero_spending": (df["promedio_gasto_comida"] == 0).sum(),
    "positive_spending": (df["promedio_gasto_comida"] > 0).sum(),
    "min_spending": df["promedio_gasto_comida"].min(),
    "max_spending": df["promedio_gasto_comida"].max()
})

missing_spending       145.00
negative_spending        0.00
zero_spending        1,463.00
positive_spending   28,392.00
min_spending             0.00
max_spending           149.97
dtype: float64

### Finding: food spending

The `promedio_gasto_comida` variable contains:

- 145 missing values
- 0 negative values
- 1,463 zero values
- 28,392 positive values
- Observed range: 0.00–149.97

All zero-spending records correspond to customers with zero visit frequency,
supporting the interpretation that these customers had no recorded restaurant
activity during the observed period.

No negative spending values are present in the current raw dataset.

**Decision:** retain zero and positive spending values. Missing spending values
will remain as `NaN` at this stage rather than being imputed without supporting
evidence.

## 2. Age validation

Age values are evaluated for missing data and implausible observations before
defining a cleaning rule.

In [21]:
pd.Series({
    "missing_age": df["edad"].isna().sum(),
    "age_zero_or_negative": (df["edad"] <= 0).sum(),
    "age_100_or_more": (df["edad"] >= 100).sum(),
    "min_age": df["edad"].min(),
    "max_age": df["edad"].max()
})

missing_age            101.00
age_zero_or_negative    97.00
age_100_or_more        110.00
min_age                 -5.00
max_age                300.00
dtype: float64

In [22]:
df.loc[
    (df["edad"] <= 0) | (df["edad"] >= 100),
    "edad"
].value_counts().sort_index()

edad
-5.00      97
300.00    110
Name: count, dtype: int64

### Finding: invalid age values

The dataset contains two clearly implausible age values:

- 97 records with age = -5
- 110 records with age = 300

These 207 observations cannot represent valid customer ages and there is no
reliable information available to reconstruct their true values.

The dataset also contains missing age values.

**Decision:** replace implausible ages (`<= 0` or `>= 100`) with `NaN`.
Customer records are retained. Missing age values will not be imputed at this
stage.

In [23]:
# Replace implausible ages with missing values
invalid_age = (df["edad"] <= 0) | (df["edad"] >= 100)

df.loc[invalid_age, "edad"] = np.nan

In [24]:
pd.Series({
    "invalid_age_after_cleaning":
        ((df["edad"] <= 0) | (df["edad"] >= 100)).sum(),

    "missing_age_after_cleaning":
        df["edad"].isna().sum(),

    "min_valid_age":
        df["edad"].min(),

    "max_valid_age":
        df["edad"].max()
})

invalid_age_after_cleaning     0.00
missing_age_after_cleaning   308.00
min_valid_age                 18.00
max_valid_age                 80.00
dtype: float64

## 3. Missing values analysis

Missing values are evaluated after the initial validity checks.

This includes both originally missing observations and values intentionally
converted to `NaN` because they were identified as invalid.

The objective is to determine whether missingness affects analytical variables,
identification fields, or both, and to define an appropriate treatment for each
case.

In [25]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2)
})

missing_report = (
    missing_report[missing_report["missing_count"] > 0]
    .sort_values("missing_pct", ascending=False)
)

missing_report

,missing_count,missing_pct
telefono_contacto,15166,50.55
correo_electronico,15072,50.24
frecuencia_visita,1547,5.16
preferencias_alimenticias,1403,4.68
edad,308,1.03
promedio_gasto_comida,145,0.48


### Missingness by city

Missing analytical values are compared across cities to identify whether
missingness is concentrated in specific markets or broadly distributed across
the dataset.

In [26]:
analytical_missing = [
    "frecuencia_visita",
    "preferencias_alimenticias",
    "edad",
    "promedio_gasto_comida"
]

missing_by_city = (
    df.groupby("ciudad_residencia")[analytical_missing]
      .apply(lambda x: x.isna().mean() * 100)
      .round(2)
)

missing_by_city

,frecuencia_visita,preferencias_alimenticias,edad,promedio_gasto_comida
ciudad_residencia,,,,
Boston,4.83,4.00,0.94,0.24
Chicago,5.31,4.98,0.97,0.65
Dallas,4.92,4.96,1.27,0.58
Denver,4.56,4.68,1.11,0.52
Houston,4.97,3.98,1.18,0.54
Miami,5.43,4.58,1.13,0.50
NYC,5.54,4.19,1.09,0.34
Phoenix,4.63,5.03,0.53,0.53
San Diego,5.33,6.18,0.91,0.46


### Finding: missingness across cities

Missing analytical values are relatively evenly distributed across the ten cities.

- Missing visit frequency ranges approximately from 4.6% to 5.5%.
- Missing food preferences range approximately from 3.9% to 6.2%.
- Missing age ranges approximately from 0.5% to 1.3%.

No strong geographic concentration of missing values is observed.

The missing values in `frecuencia_visita` correspond to the previously identified
invalid `-3` values that were intentionally converted to `NaN`.

**Decision:** city-specific deletion or imputation is not justified based on the
observed missingness patterns.

In [27]:
missing_by_segment = (
    df.groupby("estrato_socioeconomico")[analytical_missing]
      .apply(lambda x: x.isna().mean() * 100)
      .round(2)
)

missing_by_segment

,frecuencia_visita,preferencias_alimenticias,edad,promedio_gasto_comida
estrato_socioeconomico,,,,
Alto,5.19,4.74,1.02,0.63
Bajo,5.00,3.96,0.96,0.32
Medio,5.36,4.90,1.08,0.51
Muy Alto,4.93,5.00,1.02,0.37


### Finding: missingness across socioeconomic segments

Missing analytical values are also evenly distributed across socioeconomic segments.

Visit-frequency missingness remains close to 5% across all four segments,
food-preference missingness is approximately 4%–5%, and missing age values
remain close to 1%.

No socioeconomic segment shows a disproportionate concentration of missing data.

**Decision:** no segment-specific deletion or imputation strategy is required.
Missing analytical values will remain as `NaN` until the requirements of each
subsequent analysis or modeling pipeline are defined.

## 4. Categorical variables validation

Categorical variables are inspected for unexpected categories, inconsistent
labels, spelling differences, and formatting issues.

In [29]:
categorical_summary = pd.DataFrame({
    "n_unique": [df[col].nunique(dropna=True) for col in categorical_cols],
    "categories": [
        ", ".join(sorted(df[col].dropna().astype(str).unique()))
        for col in categorical_cols
    ],
    "missing": [df[col].isna().sum() for col in categorical_cols]
}, index=categorical_cols)

categorical_summary

,n_unique,categories,missing
genero,2,"Femenino, Masculino",0
ciudad_residencia,10,"Boston, Chicago, Dallas, Denver, Houston, Miam...",0
estrato_socioeconomico,4,"Alto, Bajo, Medio, Muy Alto",0
ocio,2,"No, Sí",0
consume_licor,2,"No, Sí",0
preferencias_alimenticias,6,"Carnes, Mariscos, Otro, Pescado, Vegano, Veget...",1403
membresia_premium,2,"No, Sí",0
tipo_de_pago_mas_usado,4,"App, Criptomoneda, Efectivo, Tarjeta",0


### Finding: categorical variables

Categorical variables show consistent labels and expected cardinalities.

No duplicated labels, spelling variants, capitalization inconsistencies, or
unexpected categories were identified.

`preferencias_alimenticias` is the only categorical analytical variable with
missing values (1,403 records).

**Decision:** no normalization or category replacement is required.
Missing food preferences remain as `NaN` and will be handled according to the
requirements of subsequent analysis and modeling.

## 5. Duplicate records and identifiers

The dataset is checked for fully duplicated records and duplicated customer
identifiers before producing the analytical dataset.

In [30]:
pd.Series({
    "total_records": len(df),
    "full_duplicates": df.duplicated().sum(),
    "duplicate_customer_ids": df["id_persona"].duplicated().sum(),
    "unique_customer_ids": df["id_persona"].nunique(),
    "missing_customer_ids": df["id_persona"].isna().sum()
})

total_records             30000
full_duplicates               0
duplicate_customer_ids        0
unique_customer_ids       30000
missing_customer_ids          0
dtype: int64

### Finding: duplicates and customer identifiers

The dataset contains 30,000 records and 30,000 unique customer identifiers.

No fully duplicated records, duplicated customer IDs, or missing customer IDs
were identified.

**Decision:** no records are removed. `id_persona` is retained as the unique
customer identifier but will not be used as a predictive or clustering feature.

## 6. Analytical dataset and PII exclusion

Personally identifiable information (PII) that is not required for analysis or
modeling is excluded from the analytical dataset.

The original raw dataset remains unchanged.

Excluded fields:

- `nombre`
- `apellido`
- `telefono_contacto`
- `correo_electronico`

`id_persona` is retained exclusively as a unique record identifier.

In [31]:
pii_columns = [
    "nombre",
    "apellido",
    "telefono_contacto",
    "correo_electronico"
]

df_clean = df.drop(columns=pii_columns).copy()

print(f"Raw dataset:       {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Analytical dataset: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

df_clean.head()

Raw dataset:       30,000 rows × 17 columns
Analytical dataset: 30,000 rows × 13 columns


,id_persona,edad,genero,ciudad_residencia,estrato_socioeconomico,frecuencia_visita,promedio_gasto_comida,ocio,consume_licor,preferencias_alimenticias,membresia_premium,tipo_de_pago_mas_usado,ingresos_mensuales
0,2550327378,31.00,Masculino,Miami,Alto,6.00,67.51,Sí,No,Vegetariano,Sí,Efectivo,6425
1,9446112038,40.00,Femenino,Denver,Medio,2.00,44.92,Sí,Sí,Mariscos,No,Efectivo,2374
2,3098363243,62.00,Femenino,Denver,Bajo,2.00,9.24,Sí,Sí,Vegetariano,No,Efectivo,1110
3,4013002847,41.00,Masculino,Boston,Alto,5.00,30.74,Sí,Sí,Carnes,Sí,Tarjeta,6931
4,7372911048,49.00,Femenino,San Diego,Bajo,0.00,0.00,No,No,Carnes,No,Tarjeta,1350


## 7. Final data quality validation

The cleaned analytical dataset is validated against the data-quality rules
defined during the audit.

In [32]:
quality_checks = pd.Series({
    "30,000 records retained":
        len(df_clean) == 30000,

    "customer IDs are unique":
        df_clean["id_persona"].is_unique,

    "no missing customer IDs":
        df_clean["id_persona"].isna().sum() == 0,

    "no negative visit frequency":
        (df_clean["frecuencia_visita"].dropna() < 0).sum() == 0,

    "zero frequency retained":
        (df_clean["frecuencia_visita"] == 0).sum() == 1463,

    "no invalid ages":
        (
            (df_clean["edad"].dropna() <= 0) |
            (df_clean["edad"].dropna() >= 100)
        ).sum() == 0,

    "no negative spending":
        (df_clean["promedio_gasto_comida"].dropna() < 0).sum() == 0,

    "no full duplicates":
        df_clean.duplicated().sum() == 0,

    "PII removed":
        all(col not in df_clean.columns for col in pii_columns)
})

quality_checks

30,000 records retained        True
customer IDs are unique        True
no missing customer IDs        True
no negative visit frequency    True
zero frequency retained        True
no invalid ages                True
no negative spending           True
no full duplicates             True
PII removed                    True
dtype: bool

## 8. Data Quality Report

The following report summarizes the main issues identified in the raw dataset
and the treatment applied during the data-quality process.

In [33]:
data_quality_report = pd.DataFrame({
    "issue": [
        "Full duplicate records",
        "Duplicate customer IDs",
        "Invalid negative visit frequency",
        "Zero visit frequency",
        "Invalid age values",
        "Missing age",
        "Missing food spending",
        "Missing food preferences",
        "Missing phone",
        "Missing email"
    ],
    "raw_count": [
        df_raw.duplicated().sum(),
        df_raw["id_persona"].duplicated().sum(),
        (df_raw["frecuencia_visita"] < 0).sum(),
        (df_raw["frecuencia_visita"] == 0).sum(),
        ((df_raw["edad"] <= 0) | (df_raw["edad"] >= 100)).sum(),
        df_raw["edad"].isna().sum(),
        df_raw["promedio_gasto_comida"].isna().sum(),
        df_raw["preferencias_alimenticias"].isna().sum(),
        df_raw["telefono_contacto"].isna().sum(),
        df_raw["correo_electronico"].isna().sum()
    ],
    "treatment": [
        "No action required",
        "No action required",
        "Converted to NaN; records retained",
        "Retained as valid inactive customers",
        "Converted to NaN; records retained",
        "Retained as NaN",
        "Retained as NaN",
        "Retained as NaN",
        "Excluded with PII fields",
        "Excluded with PII fields"
    ]
})

data_quality_report

,issue,raw_count,treatment
0,Full duplicate records,0,No action required
1,Duplicate customer IDs,0,No action required
2,Invalid negative visit frequency,1547,Converted to NaN; records retained
3,Zero visit frequency,1463,Retained as valid inactive customers
4,Invalid age values,207,Converted to NaN; records retained
5,Missing age,101,Retained as NaN
6,Missing food spending,145,Retained as NaN
7,Missing food preferences,1403,Retained as NaN
8,Missing phone,15166,Excluded with PII fields
9,Missing email,15072,Excluded with PII fields


In [34]:
PROCESSED_PATH = Path("../data/processed/customers_clean.csv")

df_clean.to_csv(
    PROCESSED_PATH,
    index=False,
    encoding="utf-8"
)

print(f"Clean dataset exported to: {PROCESSED_PATH}")
print(f"Final shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

Clean dataset exported to: ..\data\processed\customers_clean.csv
Final shape: 30,000 rows × 13 columns


## Conclusion

The data-quality process retained all 30,000 customer records while correcting
clearly invalid observations and removing unnecessary personally identifiable
information from the analytical dataset.

Key decisions included:

- Zero visit frequency was retained because all corresponding customers also
  showed zero food spending, indicating a consistent inactive-customer profile.
- Negative visit frequency (`-3`) was not converted to `3`, because customer
  spending, income, and premium-membership patterns did not support a simple
  sign-entry correction. These values were therefore treated as unknown.
- Implausible ages (`-5` and `300`) were converted to missing values rather than
  artificially corrected.
- Missing analytical values were retained without arbitrary imputation.
- Personally identifiable fields were excluded from the analytical dataset.
- No duplicate records or customer identifiers were found.

The resulting dataset is ready for multi-city exploratory analysis. Missing-value
treatment required for machine-learning models will be handled later within the
appropriate modeling pipeline.